In [1]:
import os, torch
from google.colab import drive
from transformers import AutoTokenizer, GPT2LMHeadModel

# load model from drive checkpoint
if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")

MODEL_DIR = "/content/drive/MyDrive/recipe_gpt2/model_250k_ingfirst"
BOS, EOS = "<|startofrecipe|>", "<|endofrecipe|>"

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = GPT2LMHeadModel.from_pretrained(MODEL_DIR)
model.eval()
if torch.cuda.is_available():
    model.to("cuda")
print("Model loaded from", MODEL_DIR)

def make_recipe(ingredients, temperature=0.7, top_p=0.9, max_length=512):
    # Sort + lowercase to match training
    norm = ", ".join(sorted(i.strip().lower() for i in ingredients.split(",") if i.strip()))
    prompt = f"{BOS}<|ingredients|>{norm}<|title|>"   # ends at title -> model writes the rest
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            inputs["input_ids"], attention_mask=inputs["attention_mask"],
            max_length=max_length, do_sample=True, temperature=temperature, top_p=top_p,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.convert_tokens_to_ids(EOS),
        )
    text = tokenizer.decode(out[0], skip_special_tokens=False)
    print(text.replace(BOS, "").replace(EOS, "").replace("<|pad|>", "")
              .replace("<|ingredients|>", "INGREDIENTS: ")
              .replace("<|title|>", "\nTITLE: ")
              .replace("<|directions|>", "\nDIRECTIONS: ").strip())

Mounted at /content/drive


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Model loaded from /content/drive/MyDrive/recipe_gpt2/model_250k_ingfirst


In [3]:
make_recipe("chicken, mango, lime, tortillas")

INGREDIENTS: chicken, lime, mango, tortillas
TITLE: Mango Chicken Tacos
DIRECTIONS: Take a large pan and put your chicken in. Make sure it is covered. Cook on medium heat for 15-20 minutes. Then cut your mango into wedges and put on tortillas.


In [8]:
make_recipe("chicken, beef, pork")

INGREDIENTS: beef, chicken, pork
TITLE: Pork And Pork
DIRECTIONS: Cook pork in skillet with small amount of water. When pork is tender, remove from pan and add 1 tablespoon of oil. Put pork back into pan and cook until done. Add chicken and sauce. Cook until sauce is thick. Serve over rice.


In [10]:
make_recipe("bacon, chili powder, chocolate, maple syrup, milk, sugar, watermelon")

INGREDIENTS: bacon, chili powder, chocolate, maple syrup, milk, sugar, watermelon
TITLE: Watermelon Bacon
DIRECTIONS: Wrap watermelon in a spiral pattern and slice into strips. Put in a saucepan with the bacon. Add sugar, and water to cover. Cook on low heat for 20 minutes. Add chili powder and maple syrup. Serve on a slice of toast.


In [11]:
make_recipe("rice, tofu, eggs, salt, pepper")

INGREDIENTS: eggs, pepper, rice, salt, tofu
TITLE: My Mother'S Tofu Fried Rice
DIRECTIONS: Start with a pan of water, and bring to a boil. Cut the tofu into thin slices, and then cut into thin strips. Mix the tofu with the rice. Mix in the egg and season with salt and pepper. Heat a pan and add 1 tablespoon of oil, and then add the tofu. Stir-fry the tofu. When the tofu is brown, add the egg and stir-fry. When the egg is cooked through, add the pepper and stir-fry until the egg is cooked through.


In [15]:
make_recipe("shrimp, lettuce, tomato, cucumber, rice, syrup")

INGREDIENTS: cucumber, lettuce, rice, shrimp, syrup, tomato
TITLE: Shrimp Rolls
DIRECTIONS: Cook rice. Boil shrimp. Drain water from shrimp. Put rice in water. Boil until done. Add shrimp. Cook. Add syrup and salt. Mix all ingredients together. Place on lettuce.
